# 2. Train from the prepared SfM Dataset — T4 ×2
Requires the output of `01_prepare_sfm100_cpu.ipynb` attached as a Kaggle Dataset, plus Oxford/Paris, Caffe weights and the existing resume checkpoint input. Set `PREPARED_DIR` in the input-discovery cell if automatic discovery is ambiguous. No subset creation or image copying runs here.

Changes: bounded same-shape microbatches (default 2, no padding), one extraction per unique mining image, one bulk GPU-to-CPU transfer for hard-negative selection, validation descriptor reuse while weights are fixed, bounded workers, sampled preflight and stage timings. Full image validation runs in the CPU preparation notebook. Missing/corrupt unsampled images can still fail later during loading.

FP32, image preprocessing, tuple draw order, distinct-cluster negative rule, summed per-tuple gradients, optimizer-step schedule and benchmark frequency remain. Batching can change floating-point rounding and near-tie rankings: this is a new pipeline run, not a claim of bitwise reproduction. Historical checkpoints are accepted subject to existing identity checks. New checkpoints record pipeline source/settings and require those to match on later resumes. GPU memory and speed require Kaggle smoke testing; no automatic precision/resolution fallback occurs.


In [ ]:
from pathlib import Path
import zipfile
import torch

# Folder checkpoint đã được Kaggle giải nén
checkpoint_dir = Path(
    '/kaggle/input/models/linh2709/seg-ep7/pytorch/default/1/latest.pt'
)

assert checkpoint_dir.is_dir(), f"Không tìm thấy folder: {checkpoint_dir}"
assert (checkpoint_dir / 'data.pkl').exists(), "Thiếu data.pkl"

# Đóng gói lại theo cấu trúc ZIP mà PyTorch yêu cầu
resume_path = Path('/kaggle/working/latest_resume.pt')

with zipfile.ZipFile(resume_path, 'w', compression=zipfile.ZIP_STORED) as z:
    for f in checkpoint_dir.rglob('*'):
        if f.is_file():
            # Đặt toàn bộ file bên trong một thư mục gốc
            arcname = f"latest.pt/{f.relative_to(checkpoint_dir)}"
            z.write(f, arcname=arcname)

print("Đã đóng gói:", resume_path)
print("Kích thước:", round(resume_path.stat().st_size / 1024**2, 2), "MB")

# Thử đọc checkpoint
state = torch.load(
    resume_path,
    map_location='cpu',
    weights_only=False
)

print("Epoch đã hoàn thành:", state['epoch'])
print("Định dạng:", state.get('format'))
del state  # Do not retain an extra CPU model/optimizer checkpoint for the whole run.

## 1. Environment and GPU checks
Use Kaggle's preinstalled compatible torch/torchvision pair. **Do not install the historical requirements.txt or run pip install .** A CUDA GPU is required by the original implementation. Two visible GPUs are required; the feature stack is split across cuda:0 and cuda:1. Outputs are confined to a notebook-owned directory under /kaggle/working.


In [ ]:
# Session logging starts BEFORE dependency imports and data discovery.
import sys, time, threading, traceback, re
from pathlib import Path
from datetime import datetime

# Safe to rerun this cell: close the previous logger and remove its hooks.
if '_session_log_stop' in globals():
    _session_log_stop()

_session_dir = Path('/kaggle/working/b0_sfm100_notebook/logs')
_session_dir.mkdir(parents=True, exist_ok=True)
SESSION_LOG = _session_dir / ('session_' + str(time.time_ns()) + '.log')
_session_file = SESSION_LOG.open('a', buffering=65536, encoding='utf-8')
_session_stdout, _session_stderr = sys.stdout, sys.stderr
_session_lock = threading.RLock()
_session_event = threading.Event()
_session_state = {'stage': 'startup', 'since': time.monotonic(), 'active': True}

# Raw writes go to disk; only complete, filtered lines reach the notebook UI.
CONSOLE_PROGRESS_SECONDS = 60
class SessionTee:
    def __init__(self, console):
        self.console = console
        self.pending = ''
        self.last_progress = float('-inf')
    def _line(self, line):
        line = line.strip('\r')
        if not line.strip():
            return
        progress = re.match(r'^(?:>>>> \d+/\d+ done|SfM paths:|Image decode preflight:|>> (?:Train|Val):)', line)
        now = time.monotonic()
        if progress:
            # Keep only the latest rate-limited progress in the UI. Full log is on disk.
            if now - self.last_progress < CONSOLE_PROGRESS_SECONDS:
                return
            self.last_progress = now
        self.console.write(line + '\n')
    def write(self, value):
        with _session_lock:
            _session_file.write(value)
            parts = re.split(r'[\r\n]', self.pending + value)
            self.pending = parts.pop()
            for line in parts:
                self._line(line)
        return len(value)
    def flush(self):
        with _session_lock:
            # A flush must not turn each print fragment into a notebook output.
            self.console.flush()
            _session_file.flush()
    def finish(self):
        with _session_lock:
            if self.pending:
                self._line(self.pending)
                self.pending = ''
            self.flush()
    def __getattr__(self, name): return getattr(self.console, name)

sys.stdout, sys.stderr = SessionTee(_session_stdout), SessionTee(_session_stderr)

def stage(message):
    _session_state.update(stage=message, since=time.monotonic(), active=True)
    print(f'[{datetime.now().isoformat(timespec="seconds")}] {message}', flush=True)

def _session_heartbeat():
    while not _session_event.wait(60):
        if _session_state['active']:
            elapsed = time.monotonic() - _session_state['since']
            print(f'[heartbeat] {_session_state["stage"]} | {elapsed:.0f}s since stage start '
                  '(kernel alive; this does not prove GPU progress)', flush=True)

def _session_pre_cell(info):
    stage('Cell started: ' + info.raw_cell.strip().splitlines()[0][:120])

def _session_post_cell(result):
    error = result.error_before_exec or result.error_in_exec
    if error is not None:
        print('[CELL FAILED] Full traceback follows:', flush=True)
        traceback.print_exception(type(error), error, error.__traceback__)
    else:
        print('[CELL COMPLETE]', flush=True)
    sys.stdout.flush(); sys.stderr.flush()
    _session_state['active'] = False

_session_ip = get_ipython()
_session_ip.events.register('pre_run_cell', _session_pre_cell)
_session_ip.events.register('post_run_cell', _session_post_cell)
_session_thread = threading.Thread(target=_session_heartbeat, daemon=True)
_session_thread.start()

def _session_log_stop():
    _session_event.set()
    _session_thread.join(timeout=2)
    _session_ip.events.unregister('pre_run_cell', _session_pre_cell)
    _session_ip.events.unregister('post_run_cell', _session_post_cell)
    sys.stdout.finish(); sys.stderr.finish()
    sys.stdout, sys.stderr = _session_stdout, _session_stderr
    _session_file.close()

stage('Session log: ' + str(SESSION_LOG))
stage('Importing dependencies and checking GPUs')

import os, sys, json, pickle, random, math, time, hashlib, base64, zlib, re, gc
from pathlib import Path
from contextlib import contextmanager, redirect_stdout, redirect_stderr
from types import SimpleNamespace
import inspect, traceback
if sys.version_info < (3, 9):
    raise RuntimeError('Python >=3.9 is required for Path.is_relative_to and str.removeprefix.')
try:
    import numpy as np
    import PIL
    from PIL import Image
    import torch
    import torchvision
    from torchvision import transforms
except (ImportError, RuntimeError, OSError) as error:
    raise RuntimeError('Kaggle dependency import failed. Use a compatible preinstalled torch/torchvision pair; '
                       'do not install the historical repository requirements. Details: ' + str(error)) from error
required_apis = {
    'Pillow Image.Resampling.LANCZOS': hasattr(Image, 'Resampling') and hasattr(Image.Resampling, 'LANCZOS'),
    'NumPy isin': callable(getattr(np, 'isin', None)),
    'torch.load(weights_only=...)': 'weights_only' in inspect.signature(torch.load).parameters,
    'Optimizer.register_step_pre_hook': callable(getattr(torch.optim.Optimizer, 'register_step_pre_hook', None)),
    'torchvision VGG16 constructor': callable(getattr(torchvision.models, 'vgg16', None)),
    'torchvision image backend API': callable(getattr(torchvision, 'get_image_backend', None)),
}
missing = [name for name, present in required_apis.items() if not present]
if missing:
    raise RuntimeError('Missing required compatibility APIs: ' + ', '.join(missing))
torchvision.set_image_backend('PIL')
print('Dependency/API preflight passed:', required_apis)

INPUT_ROOT = Path('/kaggle/input')
WORKING = Path('/kaggle/working')
if not INPUT_ROOT.is_dir() or not WORKING.is_dir():
    raise RuntimeError('Run on Kaggle with the three datasets attached and a GPU enabled.')
if not torch.cuda.is_available():
    raise RuntimeError('Enable a Kaggle GPU accelerator before continuing.')
if torch.cuda.device_count() < 2:
    raise RuntimeError('Select Kaggle GPU T4 x2 (two visible CUDA devices).')
torch.cuda.set_device(0)  # Original loader, loss, outputs and optimizer remain on cuda:0.
ENVIRONMENT = dict(python=sys.version, torch=torch.__version__, torchvision=torchvision.__version__,
                   numpy=np.__version__, pillow=PIL.__version__, cuda=torch.version.cuda,
                   gpu=torch.cuda.get_device_name(0), gpu_bytes=torch.cuda.get_device_properties(0).total_memory)
print(json.dumps(ENVIRONMENT, indent=2))
print('Devices:', torch.cuda.device_count(), 'VGG16 feature model-parallel on cuda:0 and cuda:1')
print('GPU 0:', torch.cuda.get_device_name(0), 'GPU 1:', torch.cuda.get_device_name(1))
print('Free working disk GB:', __import__('shutil').disk_usage(WORKING).free / 1e9)
WORKSPACE = WORKING / 'b0_sfm100_notebook'
WORKSPACE.mkdir(exist_ok=True)

# Shared logger is defined before smoke; all run logs stay inside the notebook workspace.
class Tee:
    def __init__(self, *streams): self.streams = streams
    def write(self, value):
        for stream in self.streams:
            stream.write(value)
        return len(value)
    def flush(self):
        for stream in self.streams: stream.flush()

@contextmanager
def logged(path):
    with Path(path).open('a', buffering=65536) as log:
        with redirect_stdout(Tee(sys.stdout, log)), redirect_stderr(Tee(sys.stderr, log)):
            yield


## Pinned source and explicit runtime adapters
Source files and hashes remain intact; runtime replacements are recorded separately.


In [ ]:
SOURCE_COMMIT = '747166e627e514842f5c5be531f4e0f73612cc90'
SOURCE_HASHES = {'cirtorch/__init__.py': 'ef6a810989ab31e2d49446d7faaa9c735f85d2fccfdedafb531f41a85b5841b5', 'cirtorch/datasets/__init__.py': 'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855', 'cirtorch/datasets/datahelpers.py': '6e18ba2c48ba66f000d83e94187f5a258862d4f490c73de1f94481da0ef811b3', 'cirtorch/datasets/genericdataset.py': '76bb2bab4493f5cfcda0691006442178727c7625abcc2e4e8ab40d6561d332d3', 'cirtorch/datasets/testdataset.py': '13c305c090657ca5624615c72ae2e8eaea26995632da9169d0b379728de117f3', 'cirtorch/datasets/traindataset.py': '2e79fa6a05ac4ee153fab383af438f6d092585205ba6ed8b1a86ab20266da702', 'cirtorch/examples/__init__.py': 'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855', 'cirtorch/examples/example_descriptor_extraction.py': '6f408acfa97edee2da6d338ff59d8bab2828a079c7b9edd7a969b51dcc5aac8f', 'cirtorch/examples/test.py': '56bc6037a7e9e91873ae3f64aefeab46f983bee1966266e241346617b52b3f7c', 'cirtorch/examples/test_e2e.py': 'e7ec88802bfba68878df91fbfb617ba0b9d3e224f3689b24b554f24d4e380016', 'cirtorch/examples/train.py': '7f0cbfd34c8075139e6225ec7db72312b97c5a0f2d7762ace9c473c928f33848', 'cirtorch/layers/__init__.py': 'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855', 'cirtorch/layers/functional.py': 'c016ffb064d458c920c61add3f6d03f1f11ae2e89d1e06facf5854636e41019c', 'cirtorch/layers/loss.py': 'fbaa3d82e297cb77481f93fe49de10af30158a0f94f487dc80bca5888f82715d', 'cirtorch/layers/normalization.py': '9ae607b73af5b0112c83a38db26586447e95f92359a3ca7eb7c5fefa3340f076', 'cirtorch/layers/pooling.py': 'c22d6d6b72e45d7f2454e6ea1b263b7ad45f548b8e50b1914fcce92bd74f8c64', 'cirtorch/networks/__init__.py': 'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855', 'cirtorch/networks/imageretrievalnet.py': 'ffc64fca4c1077987f40640eec90500f7f26b80f3a37703432572f21e8288a60', 'cirtorch/utils/__init__.py': 'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855', 'cirtorch/utils/download.py': 'd5bf645ec5417457b4062dbf010cc8388d3310d779cbe70beffd3e8000e2b195', 'cirtorch/utils/download_win.py': '643261a89fe1ceaaff19cb9c5244af178d4dc833542f87e51042a0f09e3aba19', 'cirtorch/utils/evaluate.py': 'ced1d110dd19f3f887e75f18ac9120aaff8900f5d6b52dc8c45c66c823141fd0', 'cirtorch/utils/general.py': '626e9f916199f7906926485c60b0a197e2ab112ab74cc66393ed5e54a295ea21', 'cirtorch/utils/whiten.py': '05162c86e6d8deea75d0d814d5469fa660e3006d2c98f6193a441dd08fb50dc1', 'LICENSE': 'c31c66c3945689e8876c9bbdb360228d13e4a15d84a77452dbe2db0ab67558e7'}
SOURCE_PAYLOAD = '...'
REPO_ROOT = WORKSPACE / ('source_' + SOURCE_COMMIT[:12])
source_files = json.loads(zlib.decompress(base64.b64decode(SOURCE_PAYLOAD)))
assert set(source_files) == set(SOURCE_HASHES)
for relative, contents in source_files.items():
    rel = Path(relative)
    if rel.is_absolute() or '..' in rel.parts or not (relative == 'LICENSE' or relative.startswith('cirtorch/')):
        raise RuntimeError('Unsafe embedded source path: ' + relative)
    encoded = contents.encode()
    assert hashlib.sha256(encoded).hexdigest() == SOURCE_HASHES[relative]
    dest = REPO_ROOT / rel
    if dest.exists():
        if dest.is_symlink() or dest.read_bytes() != encoded:
            raise RuntimeError('Existing source differs: ' + str(dest))
    else:
        dest.parent.mkdir(parents=True, exist_ok=True)
        dest.write_bytes(encoded)
if 'cirtorch' in sys.modules:
    imported = Path(sys.modules['cirtorch'].__file__).resolve()
    if not imported.is_relative_to(REPO_ROOT.resolve()):
        raise RuntimeError('Restart kernel: cirtorch is already imported from ' + str(imported))
sys.path.insert(0, str(REPO_ROOT))
import cirtorch
from cirtorch.datasets import traindataset, datahelpers
from cirtorch.datasets.testdataset import configdataset
from cirtorch.networks import imageretrievalnet
from cirtorch.layers.loss import ContrastiveLoss
from cirtorch.layers.pooling import GeM
from cirtorch.utils import general
from cirtorch.utils.evaluate import compute_ap, compute_map
from cirtorch.examples import train as original_train
assert Path(cirtorch.__file__).resolve().is_relative_to(REPO_ROOT.resolve())
print('Pinned source:', SOURCE_COMMIT, cirtorch.__file__)


In [ ]:
# Set exact FILE paths here if bounded discovery reports missing/ambiguous files.
# Image directories default to metadata_parent/ims and ground_truth_parent/jpg.
PATH_OVERRIDES = dict(sfm_metadata=None, sfm_images=None, oxford_ground_truth=None,
                      oxford_images=None, paris_ground_truth=None, paris_images=None, weights=None)

def input_path(p):
    p = Path(p).resolve()
    if not p.is_relative_to(INPUT_ROOT.resolve()) or not p.exists():
        raise RuntimeError('Expected existing mounted input: ' + str(p))
    return p

WANTED_FILES = {
    'sfm_metadata': 'retrieval-SfM-120k.pkl',
    'oxford_ground_truth': 'gnd_oxford5k.pkl',
    'paris_ground_truth': 'gnd_paris6k.pkl',
    'weights': 'imagenet-caffe-vgg16-features-d369c8e.pth',
}

def discover_metadata(root, wanted, max_depth=7, max_dirs=200, max_entries=4000, seconds=30):
    # Do not descend into image trees or CID's two-hex-digit directories.
    from collections import deque
    pending = deque([(Path(root), 0)])
    found = {name: [] for name in wanted}
    visited = entries = 0
    started = time.monotonic()
    truncated = False
    skip = {'ims', 'jpg', 'images', 'image', 'jpeg', 'png', '__pycache__', '.git'}
    while pending:
        if visited >= max_dirs or entries >= max_entries or time.monotonic()-started >= seconds:
            truncated = True
            break
        directory, depth = pending.popleft()
        visited += 1
        if visited == 1 or visited % 20 == 0:
            print(f'Discovery: {visited} directories, {entries} entries, current={directory}', flush=True)
        for name in wanted:
            candidate = directory / name
            if candidate.is_file():
                found[name].append(candidate)
                print('Found:', candidate, flush=True)
        # Depth limit deliberately avoids scanning the contents of deep folders.
        if depth >= max_depth:
            continue
        with os.scandir(directory) as listing:
            for entry in listing:
                entries += 1
                if entries >= max_entries or time.monotonic()-started >= seconds:
                    truncated = True
                    break
                if entry.name.lower() in skip or re.fullmatch(r'[0-9a-fA-F]{2}', entry.name):
                    continue
                if entry.is_dir(follow_symlinks=False):
                    pending.append((Path(entry.path), depth+1))
        if truncated:
            break
    print(f'Discovery finished: dirs={visited}, entries={entries}, '
          f'seconds={time.monotonic()-started:.1f}, budget_reached={truncated}', flush=True)
    return found, truncated

# Exact attached prepared folder, or None for bounded discovery.
PREPARED_DIR = None
if PREPARED_DIR is None:
    matches, truncated = discover_metadata(INPUT_ROOT, ['prepared_manifest.json'])
    if truncated or len(matches['prepared_manifest.json']) != 1:
        raise RuntimeError('Attach the prepared SfM Dataset and set PREPARED_DIR to its exact folder.')
    PREPARED_DIR = matches['prepared_manifest.json'][0].parent
PREPARED_DIR = input_path(PREPARED_DIR)
PREPARED_MANIFEST = json.loads((PREPARED_DIR / 'prepared_manifest.json').read_text())
assert PREPARED_MANIFEST['format'] == 'b0_prepared_subset_v1'
assert PREPARED_MANIFEST['full_image_decode_preflight'] is True
for filename, expected_hash in PREPARED_MANIFEST['metadata_files'].items():
    assert filename in {'retrieval-SfM-120k.pkl', 'needed_files.txt', 'subset_provenance_k100_seed42.txt'}
    actual_hash = hashlib.sha256((PREPARED_DIR / filename).read_bytes()).hexdigest()
    assert actual_hash == expected_hash, f'Prepared metadata hash mismatch: {filename}'
PATH_OVERRIDES['sfm_metadata'] = str(PREPARED_DIR / 'retrieval-SfM-120k.pkl')
PATH_OVERRIDES['sfm_images'] = str(PREPARED_DIR / 'ims')

stage('Locating metadata/weights (bounded scan; image trees skipped)')
unresolved = [name for key, name in WANTED_FILES.items() if PATH_OVERRIDES[key] is None]
file_index, discovery_truncated = discover_metadata(INPUT_ROOT, unresolved) if unresolved else ({}, False)

def choose(key, candidates):
    override = PATH_OVERRIDES[key]
    if override is not None:
        p = input_path(override)
        expected_name = WANTED_FILES.get(key)
        if not p.is_file() or (expected_name and p.name != expected_name):
            raise RuntimeError(f'{key}: expected file named {expected_name}: {p}')
        return p
    candidates = sorted(set(input_path(p) for p in candidates))
    if discovery_truncated or len(candidates) != 1:
        raise RuntimeError(f'{key}: bounded discovery found {candidates}; budget_reached={discovery_truncated}. '
                           'Set exact PATH_OVERRIDES file paths above and rerun this cell. '
                           'Discovery skips image folders and searches only 7 directory levels.')
    return candidates[0]

SFM_PKL = choose('sfm_metadata', file_index.get('retrieval-SfM-120k.pkl', []))
assert SFM_PKL.name == 'retrieval-SfM-120k.pkl'
with SFM_PKL.open('rb') as f:
    SFM_DB = pickle.load(f)  # Trusted team-supplied mounted metadata only.
PROVENANCE_PATH = SFM_PKL.parent / 'subset_provenance_k100_seed42.txt'
FILTER_PATH = SFM_PKL.parent / 'needed_files.txt'
# Generated provenance/filter are written after subset selection below.
SOURCE_SFM_PKL = SFM_PKL
WEIGHTS = choose('weights', file_index.get('imagenet-caffe-vgg16-features-d369c8e.pth', []))
print('Active metadata:', SFM_PKL)
print('Prepared subset loaded; no full-data sampling or image export in training.')


In [ ]:
# Reuse already loaded metadata: no second recursive scan of /kaggle/input.
stage('Inspecting selected SfM metadata: ' + str(SOURCE_SFM_PKL))
for split in ['train', 'val']:
    d = SFM_DB[split]
    print(split, 'images:', len(d['cids']), 'clusters:', len(set(d['cluster'])),
          'pairs:', len(d['qidxs']), flush=True)


In [ ]:
EXPECTED_SUBSET = {
    'train': (15953, 100, 34337),
    'val': (739, 20, 181)
}
assert isinstance(SFM_DB, dict) and all(part in SFM_DB for part in EXPECTED_SUBSET)
SUBSET_SUMMARY = {}
for part, expected in EXPECTED_SUBSET.items():
    d = SFM_DB[part]
    assert all(k in d for k in ['cids', 'cluster', 'qidxs', 'pidxs']), part
    cids, clusters, qs, ps = (d[k] for k in ['cids', 'cluster', 'qidxs', 'pidxs'])
    assert len(cids) == len(clusters) and len(qs) == len(ps), part
    actual = (len(cids), len(set(clusters)), len(qs))
    assert actual == expected, f'{part}: expected {expected}, got {actual}; refuse a different subset'
    assert len(set(cids)) == len(cids)
    assert all(isinstance(cid, str) and len(cid) >= 6 and Path(cid).name == cid
               and cid not in {'.', '..'} for cid in cids)
    assert all(isinstance(i, (int, np.integer)) and 0 <= i < len(cids) for i in list(qs) + list(ps))
    assert all(q != p and clusters[q] == clusters[p] for q, p in zip(qs, ps)), part
    SUBSET_SUMMARY[part] = dict(images=actual[0], clusters=actual[1], pairs=actual[2],
                                cluster_ids=sorted(int(c) if isinstance(c, np.integer) else c for c in set(clusters)))
assert not set(SFM_DB['train']['cids']) & set(SFM_DB['val']['cids']), 'CID leakage'
assert not set(SFM_DB['train']['cluster']) & set(SFM_DB['val']['cluster']), 'Cluster leakage'
ALL_CIDS = list(SFM_DB['train']['cids']) + list(SFM_DB['val']['cids'])
assert len(ALL_CIDS) == len(set(ALL_CIDS)) == 16692
provenance_text = PROVENANCE_PATH.read_text()
print(provenance_text)
assert re.search(r'num_clusters\s*=\s*100\b', provenance_text)
assert re.search(r'seed\s*=\s*42\b', provenance_text)
for part, (ni, nc, npairs) in EXPECTED_SUBSET.items():
    assert re.search(rf'{part}:\s*{nc}\s+clusters,\s*{ni:,}\s+images,\s*{npairs:,}\s+pairs', provenance_text)
expected_filter = {f'{cid[-2:]}/{cid[-4:-2]}/{cid[-6:-4]}/{cid}' for cid in ALL_CIDS}
actual_filter = {line.removeprefix('./') for line in FILTER_PATH.read_text().splitlines() if line.strip()}
assert actual_filter == expected_filter, 'needed_files.txt differs from active metadata'

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

DATASET_IDENTITY = dict(subset_seed=42, partitions=SUBSET_SUMMARY,
                       metadata_sha256=sha256_file(SFM_PKL), provenance_sha256=sha256_file(PROVENANCE_PATH),
                       filter_sha256=sha256_file(FILTER_PATH), weights_sha256=sha256_file(WEIGHTS))
print(json.dumps(SUBSET_SUMMARY, indent=2))

assert DATASET_IDENTITY == PREPARED_MANIFEST['dataset_identity'], 'Prepared data/weights identity differs'
assert expected_filter == set(PREPARED_MANIFEST['sfm_images']), 'Prepared image inventory differs'
# Sample integrity without a repeated 16k-file scan/decode. Use an independent RNG.
for relative in random.Random(42).sample(sorted(expected_filter), min(64,len(expected_filter))):
    path = PREPARED_DIR / 'ims' / relative
    assert sha256_file(path) == PREPARED_MANIFEST['sfm_images'][relative]['sha256'], str(path)


## 5. Validate image paths and benchmark inputs
Check all referenced SfM and benchmark paths, ranking indices and nonempty query crop overlap; fully decode images on CPU before GPU work. This adds startup time but exposes missing or corrupt files before mining. No image content checksum is claimed.


In [ ]:
for name, key, expected in [
    ("oxford5k", "oxford", 5063),
    ("paris6k", "paris", 6392)
]:
    gp = choose(
        key + "_ground_truth",
        file_index.get("gnd_" + name + ".pkl", [])
    )

    with gp.open("rb") as f:
        gt = pickle.load(f)

    print(f"\n{name.upper()}")
    print("Metadata:", gp)
    print("Expected images:", expected)
    print("Actual images:", len(gt["imlist"]))
    print("Unique images:", len(set(gt["imlist"])))
    print("Queries:", len(gt["qimlist"]))
    print("Ground truth entries:", len(gt["gnd"]))

In [ ]:

# ============================================================
# CELL 11 — SfM subset-100, benchmarks, offline scaffold
# ============================================================

from pathlib import Path
import pickle
import hashlib
import json

# ------------------------------------------------------------
# 0. Explicit paths — independent of old Editor variables
# ------------------------------------------------------------

# Reuse the validated metadata/weights and honor PATH_OVERRIDES.
DATA_ROOT = WORKSPACE / 'data'
SFM_PKL = Path(SFM_PKL).resolve()
SFM_IMAGES = input_path(PATH_OVERRIDES['sfm_images'] or (SOURCE_SFM_PKL.parent / 'ims'))
WEIGHTS = Path(WEIGHTS).resolve()

assert SFM_PKL.is_file(), f"Missing metadata: {SFM_PKL}"
assert SFM_IMAGES.is_dir(), f"Missing images: {SFM_IMAGES}"
assert WEIGHTS.is_file(), f"Missing weights: {WEIGHTS}"

# ------------------------------------------------------------
# 1. Validate generated SfM subset-100
# ------------------------------------------------------------

with SFM_PKL.open("rb") as f:
    active_sfm_db = pickle.load(f)

train = active_sfm_db["train"]
val = active_sfm_db["val"]

assert len(train["cids"]) == 15953
assert len(set(train["cluster"])) == 100
assert len(train["qidxs"]) == 34337
assert len(train["pidxs"]) == 34337

assert len(val["cids"]) == 739
assert len(set(val["cluster"])) == 20
assert len(val["qidxs"]) == 181
assert len(val["pidxs"]) == 181

for partition_name, partition in [
    ("train", train),
    ("val", val)
]:
    n = len(partition["cids"])
    clusters = partition["cluster"]

    for q, p in zip(
        partition["qidxs"],
        partition["pidxs"]
    ):
        q, p = int(q), int(p)

        assert 0 <= q < n
        assert 0 <= p < n
        assert q != p
        assert clusters[q] == clusters[p]

    print(
        f"PASS {partition_name}: "
        f"{n} images, "
        f"{len(set(clusters))} clusters, "
        f"{len(partition['qidxs'])} pairs"
    )

# Check the exact CID layout used by the original loader, before any GPU mining.
referenced = [Path(datahelpers.cid2filename(cid, str(SFM_IMAGES)))
              for part in ['train', 'val'] for cid in active_sfm_db[part]['cids']]
stage('Checking SfM image paths')
missing_images = []
sampled_references = random.Random(42).sample(referenced, min(64, len(referenced)))
for index, path in enumerate(sampled_references, 1):
    if not path.is_file():
        missing_images.append(str(path))
    if index % 1000 == 0 or index == len(referenced):
        print(f'SfM paths: {index}/{len(referenced)}', flush=True)
if missing_images:
    raise FileNotFoundError(f'{len(missing_images)} SfM images missing; first paths: {missing_images[:10]}')
print('SfM paths sampled:', len(sampled_references), 'of', len(referenced))

# ------------------------------------------------------------
# 2. Initialize benchmarks explicitly
# ------------------------------------------------------------

BENCHMARKS = {}

for name, expected_images in [
    ("oxford5k", 5063),
    ("paris6k", 6392)
]:
    key = 'oxford' if name == 'oxford5k' else 'paris'
    gp = choose(key + '_ground_truth', file_index.get('gnd_' + name + '.pkl', []))
    jpg = input_path(PATH_OVERRIDES[key + '_images'] or (gp.parent / 'jpg'))

    assert gp.is_file(), f"Missing ground truth: {gp}"
    assert jpg.is_dir(), f"Missing image directory: {jpg}"

    with gp.open("rb") as f:
        gt = pickle.load(f)

    assert all(
        k in gt for k in ("imlist", "qimlist", "gnd")
    ), name

    assert len(gt["imlist"]) == expected_images
    assert len(set(gt["imlist"])) == expected_images
    assert len(gt["qimlist"]) == 55
    assert len(gt["gnd"]) == 55

    # Check every gallery and query image exists.
    all_ids = set(gt["imlist"]) | set(gt["qimlist"])

    for image_id in random.Random(42).sample(sorted(all_ids), min(64, len(all_ids))):
        assert isinstance(image_id, str)
        assert Path(image_id).name == image_id
        assert (jpg / f"{image_id}.jpg").is_file(), image_id

    # Invalid ranking indices or empty query crops otherwise fail much later.
    for query_id, ground_truth in zip(gt['qimlist'], gt['gnd']):
        for category in ['ok', 'junk']:
            indices = ground_truth.get(category, [])
            assert all(isinstance(i, (int, np.integer)) and 0 <= i < expected_images for i in indices), (name, category)
        bbox = ground_truth['bbx']
        assert len(bbox) == 4 and all(math.isfinite(float(v)) for v in bbox), (name, query_id)
        x0, y0, x1, y1 = map(float, bbox)
        with Image.open(jpg / f'{query_id}.jpg') as image:
            w, h = image.size
        assert x0 < x1 and y0 < y1 and x1 > 0 and y1 > 0 and x0 < w and y0 < h, (name, query_id, bbox)

    BENCHMARKS[name] = {
        "ground_truth": gp,
        "images": jpg,
        "gnd": gt
    }

    print(
        f"PASS {name}: "
        f"{expected_images} gallery images, "
        f"55 queries"
    )

# Full decoding catches truncated/corrupt images before training.
image_paths = list(dict.fromkeys(referenced + [b['images'] / (cid + '.jpg')
    for b in BENCHMARKS.values() for cid in set(b['gnd']['imlist']) | set(b['gnd']['qimlist'])]))
image_paths = random.Random(42).sample(image_paths, min(128, len(image_paths)))
stage(f'Sample-decoding {len(image_paths)} images; full SfM validation was done in preparation')
for index, path in enumerate(image_paths, 1):
    try:
        with Image.open(path) as image:
            image.convert('RGB').load()
    except Exception as error:
        raise RuntimeError(f'Image cannot be decoded: {path}: {error}') from error
    if index % 100 == 0 or index == len(image_paths):
        print(f'Image decode preflight: {index}/{len(image_paths)}', flush=True)

# ------------------------------------------------------------
# 3. Approved sources and safe symlinks
# ------------------------------------------------------------

def validate_source(source):
    source = Path(source).resolve()

    if not source.exists():
        raise FileNotFoundError(source)

    allowed_input = source.is_relative_to(INPUT_ROOT)

    allowed_subset = source.is_relative_to(
        (WORKSPACE / "generated_subset100").resolve()
    )

    if not (allowed_input or allowed_subset):
        raise RuntimeError(
            f"Unapproved source: {source}"
        )

    return source


def link_input(destination, source, allow_repair=False):
    destination = Path(destination)
    source = validate_source(source)

    destination.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    if destination.is_symlink():
        if destination.resolve() != source:
            if not allow_repair:
                raise RuntimeError(
                    f"Unexpected symlink: {destination}"
                )

            destination.unlink()
            destination.symlink_to(
                source,
                target_is_directory=source.is_dir()
            )

            print("Repaired:", destination)

    elif destination.exists():
        raise RuntimeError(
            f"Refusing to overwrite: {destination}"
        )

    else:
        destination.symlink_to(
            source,
            target_is_directory=source.is_dir()
        )

    assert destination.resolve() == source
    return destination

# ------------------------------------------------------------
# 4. SfM training scaffold
# ------------------------------------------------------------

sfm_stage = (
    DATA_ROOT
    / "train"
    / "retrieval-SfM-120k"
)

metadata_link = link_input(
    sfm_stage / "retrieval-SfM-120k.pkl",
    SFM_PKL,
    allow_repair=True
)

images_link = link_input(
    sfm_stage / "ims",
    SFM_IMAGES,
    allow_repair=True
)

# ------------------------------------------------------------
# 5. Benchmark scaffold
# ------------------------------------------------------------

for name, b in BENCHMARKS.items():

    link_input(
        DATA_ROOT / "test" / name / f"gnd_{name}.pkl",
        b["ground_truth"]
    )

    link_input(
        DATA_ROOT / "test" / name / "jpg",
        b["images"]
    )

# ------------------------------------------------------------
# 6. VGG16 pretrained weights
# ------------------------------------------------------------

link_input(
    DATA_ROOT / "networks" / WEIGHTS.name,
    WEIGHTS
)

# ------------------------------------------------------------
# 7. Redirect original source to workspace
# ------------------------------------------------------------

for module in [
    general,
    traindataset,
    imageretrievalnet,
    original_train
]:
    module.get_data_root = lambda: str(DATA_ROOT)

# ------------------------------------------------------------
# 8. Disable original automatic downloaders
# ------------------------------------------------------------

def reject_download(*args, **kwargs):
    raise RuntimeError(
        "Automatic downloading is disabled. "
        "Use attached Kaggle inputs."
    )

from cirtorch.utils import download as source_download

source_download.download_train = reject_download
source_download.download_test = reject_download

original_train.download_train = reject_download
original_train.download_test = reject_download
original_train.main = reject_download

imageretrievalnet.model_zoo.load_url = reject_download

# ------------------------------------------------------------
# 9. Rebuild metadata used by later cells
# ------------------------------------------------------------

def sha256_file(path):
    h = hashlib.sha256()

    with Path(path).open("rb") as f:
        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            h.update(chunk)

    return h.hexdigest()


PATHS = {
    "sfm_metadata": str(SFM_PKL),
    "sfm_images": str(SFM_IMAGES),
    "weights": str(WEIGHTS)
}

for name, b in BENCHMARKS.items():
    PATHS[name] = {
        "ground_truth": str(b["ground_truth"]),
        "images": str(b["images"])
    }

# Preserve dataset identity if an earlier cell created it.
if "DATASET_IDENTITY" not in globals():
    DATASET_IDENTITY = {}

DATASET_IDENTITY["metadata_sha256"] = sha256_file(SFM_PKL)
DATASET_IDENTITY["weights_sha256"] = sha256_file(WEIGHTS)

for name, b in BENCHMARKS.items():
    DATASET_IDENTITY[name] = {
        "ground_truth_sha256": sha256_file(
            b["ground_truth"]
        ),
        "gallery": len(b["gnd"]["imlist"]),
        "queries": len(b["gnd"]["qimlist"])
    }

# ------------------------------------------------------------
# 10. Final verification
# ------------------------------------------------------------

assert metadata_link.resolve() == SFM_PKL
assert images_link.resolve() == SFM_IMAGES

assert not (
    sfm_stage / "retrieval-SfM-120k-full.pkl"
).exists()

with metadata_link.open("rb") as f:
    verified_db = pickle.load(f)

assert len(verified_db["train"]["cids"]) == 15953
assert len(verified_db["train"]["qidxs"]) == 34337

print("\n" + "=" * 55)
print("CELL 11 PASSED")
print("=" * 55)

print("SfM metadata:", metadata_link.resolve())
print("SfM images:", images_link.resolve())

print(
    "Training images:",
    len(verified_db["train"]["cids"])
)
print(
    "Training clusters:",
    len(set(verified_db["train"]["cluster"]))
)
print(
    "Positive pairs:",
    len(verified_db["train"]["qidxs"])
)

print("\nBenchmarks:")
for name, b in BENCHMARKS.items():
    print(
        f"  {name}: "
        f"{len(b['gnd']['imlist'])} images"
    )

print("\nOffline guard: ACTIVE")
print("Original source files: UNCHANGED")
print("=" * 55)


In [ ]:
import gc
import json
import math
import random

import numpy as np
import torch
from torchvision import transforms

SOURCE_DEFAULTS = {
    'image_size': 1024,
    'query_size': 2000,
    'pool_size': 20000,
    'epochs': 100,
    'batch_size': 5,
    'update_every': 1,
    'optimizer': 'adam',
    'lr': 1e-6,
    'weight_decay': 1e-6,
    'momentum': 0.9,
    'loss_margin': 0.7,
    'gem_p': 3.0,
    'seed': 0,
    'workers': 8,
    'test_freq': 1,
    'eval_image_size': 1024,
    'scheduler': 'upstream_pre_epoch',
    'selection': 'validation_loss',
    'validation': True,
}

TRAIN_CHOICES = dict(SOURCE_DEFAULTS, seed=42)

SMOKE_MINING_POOL_CAP = 256
DECISION_NOTE = 'B0 VGG16-GeM reproduction on SfM subset-100; configuration from TRAIN_CHOICES; smoke and full training controlled separately.'

# Bản sao riêng cho full training, không còn các giá trị None.
FULL_TRAIN_CHOICES = dict(TRAIN_CHOICES)
FULL_DECISION_NOTE = DECISION_NOTE

RUN_ID = 'b0_sfm100_vgg16_gem'
RESUME_CHECKPOINT = "/kaggle/working/latest_resume.pt"
RUN_SMOKE = False
RUN_FULL_TRAINING = True
RUN_FINAL_EVALUATION = False
EVALUATION_CHECKPOINT = None


def resolved_config(purpose='smoke'):
    assert purpose in ['smoke', 'full']

    choices = TRAIN_CHOICES if purpose == 'smoke' else FULL_TRAIN_CHOICES
    note = DECISION_NOTE if purpose == 'smoke' else FULL_DECISION_NOTE
    missing = [key for key, value in choices.items() if value is None]

    if missing or not note.strip():
        raise RuntimeError(
            f'Resolve {purpose} choices={missing} '
            'and its decision note before execution.'
        )

    cfg = dict(choices)

    if purpose == 'smoke':
        assert isinstance(SMOKE_MINING_POOL_CAP, int)
        assert not isinstance(SMOKE_MINING_POOL_CAP, bool)
        assert SMOKE_MINING_POOL_CAP >= 6

        cfg.update(
            query_size=cfg['batch_size'],
            pool_size=min(SMOKE_MINING_POOL_CAP, 15953),
            update_every=1,
            epochs=1,
        )

    integer_keys = [
        'image_size', 'query_size', 'pool_size', 'epochs',
        'batch_size', 'update_every', 'seed', 'workers',
        'test_freq', 'eval_image_size',
    ]

    for key in integer_keys:
        assert isinstance(cfg[key], int), key
        assert not isinstance(cfg[key], bool), key

    assert cfg['image_size'] >= 32
    assert cfg['eval_image_size'] >= 32
    assert 1 <= cfg['batch_size'] <= cfg['query_size'] <= len(SFM_DB['train']['qidxs'])
    assert cfg['pool_size'] >= 6
    assert cfg['epochs'] > 0
    assert cfg['update_every'] > 0
    assert cfg['seed'] >= 0
    assert cfg['workers'] >= 0
    assert cfg['test_freq'] > 0
    assert cfg['optimizer'] in ['adam', 'sgd']

    for key in ['lr', 'loss_margin', 'gem_p']:
        assert math.isfinite(cfg[key]) and cfg[key] > 0, key

    assert math.isfinite(cfg['weight_decay'])
    assert cfg['weight_decay'] >= 0
    assert math.isfinite(cfg['momentum'])
    assert 0 <= cfg['momentum'] < 1

    assert cfg['scheduler'] == 'upstream_pre_epoch'
    assert cfg['selection'] == 'validation_loss'
    assert cfg['validation'] is True

    # Giữ kiểm tra accumulation của code gốc.
    assert (
        cfg['query_size'] // cfg['batch_size']
    ) % cfg['update_every'] == 0
    assert cfg['batch_size'] <= 181

    return cfg


print(
    'Full training config:',
    json.dumps(resolved_config('full'), indent=2),
)
print(
    'RUN_SMOKE =', RUN_SMOKE,
    '| RUN_FULL_TRAINING =', RUN_FULL_TRAINING,
)


def seed_all(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False


# WEIGHTS và các module/data bên dưới được chuẩn bị ở những ô trước.
stage('Loading local VGG16 weights: ' + str(WEIGHTS))
FEATURE_STATE = torch.load(
    WEIGHTS,
    map_location='cpu',
    weights_only=True,
)

assert isinstance(FEATURE_STATE, dict) and FEATURE_STATE
assert all(
    isinstance(key, str) and isinstance(value, torch.Tensor)
    for key, value in FEATURE_STATE.items()
)


class TwoT4Features(torch.nn.Sequential):
    split_at = 17

    def forward(self, x):
        for index, layer in enumerate(self):
            if index == self.split_at:
                x = x.to('cuda:1', non_blocking=True)
            x = layer(x)

        return x.to('cuda:0', non_blocking=True)


def place_model_on_two_t4(model):
    from types import MethodType

    assert torch.cuda.device_count() >= 2, 'Chọn GPU T4 x2 trên Kaggle.'

    model.features = TwoT4Features(
        *list(model.features.children())
    )
    assert len(model.features) > TwoT4Features.split_at

    for index, layer in enumerate(model.features):
        device = (
            'cuda:0'
            if index < TwoT4Features.split_at
            else 'cuda:1'
        )
        layer.to(device)

    model.pool.to('cuda:0')
    model.norm.to('cuda:0')

    if model.lwhiten is not None or model.whiten is not None:
        raise RuntimeError(
            'Unexpected whitening module in B0 device placement'
        )

    def keep_two_t4_placement(self, device=None):
        if device is not None:
            raise RuntimeError(
                'This B0 model is fixed across cuda:0 and cuda:1'
            )
        return self

    model.cuda = MethodType(keep_two_t4_placement, model)

    assert model.features[0].weight.device == torch.device('cuda:0')
    assert (
        model.features[TwoT4Features.split_at].weight.device
        == torch.device('cuda:1')
    )
    assert model.pool.p.device == torch.device('cuda:0')

    return model


def make_model(p=3.0):
    settings = {
        'architecture': 'vgg16',
        'pooling': 'gem',
        'pretrained': False,
        'local_whitening': False,
        'whitening': False,
        'regional': False,
    }

    stage('Constructing VGG16 and loading feature weights')
    model = imageretrievalnet.init_network(settings)
    model.features.load_state_dict(FEATURE_STATE, strict=True)

    with torch.no_grad():
        model.pool.p.fill_(p)

    assert isinstance(model.pool, GeM)
    assert model.pool.p.numel() == 1
    assert model.meta['outputdim'] == 512
    assert model.whiten is None
    assert model.lwhiten is None

    for key, tensor in model.features.state_dict().items():
        assert torch.equal(
            tensor.cpu(), FEATURE_STATE[key]
        ), 'Pretrained load mismatch: ' + key

    stage('Placing VGG16 features on two GPUs')
    return place_model_on_two_t4(model)


# One real, preflight-checked image; this is initialization only, not training.
def initialization_probe(cfg):
    seed_all(cfg['seed'])
    probe_model = descriptor = probe = None
    try:
        probe_model = make_model(cfg['gem_p'])
        probe_model.eval()
        preprocessing = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize(mean=probe_model.meta['mean'], std=probe_model.meta['std']),
        ])
        stage('Initialization probe: 128px forward pass')
        probe = datahelpers.imresize(datahelpers.pil_loader(str(referenced[0])), 128)
        with torch.no_grad():
            descriptor = probe_model(preprocessing(probe).unsqueeze(0).cuda())
        assert descriptor.shape == (512, 1) and torch.isfinite(descriptor).all()
        assert torch.allclose(descriptor.norm(dim=0), torch.ones(1, device='cuda:0'), atol=1e-5)
        print(probe_model.meta_repr(), 'p=', probe_model.pool.p.item(), 'descriptor=', tuple(descriptor.shape))
        return preprocessing
    finally:
        probe_model = descriptor = probe = None
        gc.collect()
        torch.cuda.empty_cache()

ACTIVE_CONFIG = resolved_config('full' if RUN_FULL_TRAINING else 'smoke')
transform = initialization_probe(ACTIVE_CONFIG)

# Both devices matter for this model-parallel network.
def reset_gpu_peaks():
    for device in range(torch.cuda.device_count()):
        torch.cuda.reset_peak_memory_stats(device)

def gpu_memory_stats():
    return {str(device): {'allocated': torch.cuda.memory_allocated(device),
                         'peak_allocated': torch.cuda.max_memory_allocated(device)}
            for device in range(torch.cuda.device_count())}


In [ ]:
# Operational settings are outside cfg so historical checkpoint identity stays valid.
# Worker scheduling can affect exact reproducibility; actual values enter run_record.json.
AVAILABLE_CPUS = len(os.sched_getaffinity(0)) if hasattr(os, 'sched_getaffinity') else (os.cpu_count() or 1)
LOADER_WORKER_CAP = min(4, AVAILABLE_CPUS)

def effective_workers(requested):
    return min(int(requested), LOADER_WORKER_CAP)

class _ModuleView:
    def __init__(self, target, **overrides):
        self._target = target
        self.__dict__.update(overrides)
    def __getattr__(self, name):
        return getattr(self._target, name)

# Mining/evaluation embed num_workers=8. Override only their module-local torch view.
_REAL_DATALOADER = torch.utils.data.DataLoader

def _bounded_loader(*args, **kwargs):
    kwargs['num_workers'] = effective_workers(kwargs.get('num_workers', 0))
    return _REAL_DATALOADER(*args, **kwargs)

for _module in (traindataset, imageretrievalnet):
    _module.torch = _ModuleView(torch, utils=_ModuleView(torch.utils,
        data=_ModuleView(torch.utils.data, DataLoader=_bounded_loader)))
print('Operational worker cap:', LOADER_WORKER_CAP, '(training, mining, evaluation)')

"""Embedded into the pipeline notebook. Uses its pinned cirtorch and torch globals.
No source-file mutation; no descriptor survives a training epoch or optimizer update.
"""
from collections import defaultdict
import torch

PIPELINE_SETTINGS = dict(version='pipeline_v2', inference_microbatch=2,
                         training_microbatch=2, max_batch_pixels=2 * 1024 * 1024,
                         extraction_window=8)


class PipelineContrastiveLoss(torch.nn.Module):
    """Original summed contrastive formula for fixed q+p+5 tuples, no GPU label scan."""
    def __init__(self, margin=0.7, eps=1e-6):
        super().__init__()
        self.margin, self.eps = margin, eps
    def forward(self, x, label):
        if x.shape[1] % 7 or label.numel() != x.shape[1]:
            raise ValueError('Expected fixed seven-image retrieval tuples.')
        tuples = x.reshape(x.shape[0], -1, 7)
        difference = tuples[:, :, :1] - tuples[:, :, 1:]
        distance = (difference + self.eps).square().sum(dim=0).sqrt()
        labels = label.reshape(-1, 7)[:, 1:]
        losses = (0.5 * labels * distance.square()
                  + 0.5 * (1-labels) * torch.clamp(self.margin-distance, min=0).square())
        return losses.sum()


def _shape_groups(images, limit):
    buckets = defaultdict(list)
    for i, image in enumerate(images):
        assert image.ndim == 4 and image.shape[0] == 1
        buckets[tuple(image.shape[1:])].append(i)
    for shape, indices in buckets.items():
        pixels = shape[-2] * shape[-1]
        size = max(1, min(limit, PIPELINE_SETTINGS['max_batch_pixels'] // pixels))
        for start in range(0, len(indices), size):
            yield indices[start:start + size]


def pipeline_tuple_vectors(model, images, limit=None):
    """Group only equal shapes, preserve output order and autograd; never pad/resize."""
    if limit is None:
        limit = PIPELINE_SETTINGS['training_microbatch']
    vectors = [None] * len(images)
    device = next(model.parameters()).device
    for indices in _shape_groups(images, limit):
        # Transfer pinned loader tensors first; a CPU torch.cat would lose pinning.
        tensors = [images[i].to(device, non_blocking=True) for i in indices]
        batch = tensors[0] if len(tensors) == 1 else torch.cat(tensors, dim=0)
        descriptors = model(batch)
        assert descriptors.shape == (model.meta['outputdim'], len(indices))
        for j, index in enumerate(indices):
            vectors[index] = descriptors[:, j:j+1]
    return torch.cat(vectors, dim=1)


def pipeline_extract_gpu(model, images, image_size, transform, bbxs=None):
    """Bounded CPU window, batch equal shapes, retain descriptors on GPU until done."""
    dataset = traindataset.ImagesFromList(root='', images=images, imsize=image_size,
                                         bbxs=bbxs, transform=transform)
    loader = torch.utils.data.DataLoader(dataset, batch_size=1, shuffle=False,
        num_workers=effective_workers(8), pin_memory=True, persistent_workers=False,
        generator=torch.Generator().manual_seed(0))
    device = next(model.parameters()).device
    result = torch.empty(model.meta['outputdim'], len(images), device=device)
    window, offset = [], 0
    with torch.no_grad():
        for image in loader:
            window.append(image)
            if len(window) == PIPELINE_SETTINGS['extraction_window']:
                vectors = pipeline_tuple_vectors(model, window, PIPELINE_SETTINGS['inference_microbatch'])
                result[:, offset:offset+len(window)] = vectors
                offset += len(window)
                window.clear()
        if window:
            vectors = pipeline_tuple_vectors(model, window, PIPELINE_SETTINGS['inference_microbatch'])
            result[:, offset:] = vectors
    return result


def pipeline_extract_vectors(net, images, image_size, transform, bbxs=None, ms=[1], msp=1, print_freq=10):
    if list(ms) != [1]:
        raise ValueError('Pipeline supports the notebook single-scale protocol only.')
    net.eval()
    torch.empty((), dtype=torch.int64).random_()  # Original extraction loader base seed.
    return pipeline_extract_gpu(net, images, image_size, transform, bbxs).cpu()


def _select_negative_indices(ranks_cpu, pool_indices, query_indices, clusters, nnum):
    """CPU-only cluster selection after one bulk rank transfer, exact ranked scan."""
    selected = []
    for q, query in enumerate(query_indices):
        seen = {clusters[query]}
        negatives = []
        for rank in ranks_cpu[:, q]:
            candidate = int(pool_indices[int(rank)])
            cluster = clusters[candidate]
            if cluster not in seen:
                negatives.append(candidate)
                seen.add(cluster)
                if len(negatives) == nnum:
                    break
        if len(negatives) != nnum:
            raise RuntimeError('Insufficient distinct negative clusters in sampled pool.')
        selected.append(negatives)
    return selected


def pipeline_mine(dataset, net):
    # Keep the upstream CPU RNG draw order exactly: pairs first, then pool.
    pair_indices = torch.randperm(len(dataset.qpool))[:dataset.qsize].tolist()
    dataset.qidxs = [dataset.qpool[i] for i in pair_indices]
    dataset.pidxs = [dataset.ppool[i] for i in pair_indices]
    pool = torch.randperm(len(dataset.images))[:dataset.poolsize].tolist()
    unique = list(dict.fromkeys(pool + dataset.qidxs))
    positions = {index: pos for pos, index in enumerate(unique)}
    net.eval()
    # Original query and pool loader iterators each consume one CPU base seed.
    for _ in range(2):
        torch.empty((), dtype=torch.int64).random_()
    with torch.no_grad():
        descriptors = pipeline_extract_gpu(net, [dataset.images[i] for i in unique],
                                           dataset.imsize, dataset.transform)
        qvecs = descriptors[:, [positions[i] for i in dataset.qidxs]]
        poolvecs = descriptors[:, [positions[i] for i in pool]]
        _, ranks = torch.sort(poolvecs.t() @ qvecs, dim=0, descending=True)
        dataset.nidxs = _select_negative_indices(ranks.cpu().numpy(), pool,
                                                 dataset.qidxs, dataset.clusters, dataset.nnum)
        negative_positions = [[positions[i] for i in ns] for ns in dataset.nidxs]
        # Original epsilon placement retained; only diagnostic reduction is vectorized.
        negatives = descriptors[:, negative_positions]
        mean_distance = (qvecs.unsqueeze(-1) - negatives + 1e-6).square().sum(dim=0).sqrt().mean().item()
        if dataset.mode == 'val':
            # Validation weights do not change between mining and validation loss.
            dataset._pipeline_validation_vectors = descriptors
            dataset._pipeline_validation_positions = positions
        dataset.last_extraction_count = len(unique)
        print('Mining extracted', len(unique), 'unique images; original separate passes:', len(pool)+len(dataset.qidxs))
    return mean_distance


def pipeline_train(loader, model, criterion, optimizer, epoch):
    loader.dataset.create_epoch_tuples(model)
    model.train()
    model.apply(original_train.set_batchnorm_eval)
    optimizer.zero_grad()
    total_loss, count = 0.0, 0
    pending_losses = []
    started = time.monotonic()
    # Worker replicas must be recreated after each epoch's tuple remine.
    for i, (images, targets) in enumerate(loader):
        for image_tuple, target in zip(images, targets):
            vectors = pipeline_tuple_vectors(model, image_tuple)
            loss = criterion(vectors, target.to(vectors.device, non_blocking=True))
            loss.backward()  # Original per-tuple backward and summed gradient accumulation.
            pending_losses.append(loss.detach())
            count += 1
        if (i+1) % original_train.args.update_every == 0:
            optimizer.step()
            optimizer.zero_grad()
        if i == 0 or (i+1) % original_train.args.print_freq == 0 or i+1 == len(loader):
            # Defer scalar host synchronization until a progress boundary.
            total_loss += torch.stack(pending_losses).double().sum().item()
            pending_losses.clear()
            print(f'>> Train: [{epoch+1}][{i+1}/{len(loader)}] loss={total_loss/count:.6f} elapsed={time.monotonic()-started:.1f}s')
    return total_loss / count


def pipeline_validate(loader, model, criterion, epoch):
    loader.dataset.create_epoch_tuples(model)
    model.eval()
    dataset = loader.dataset
    try:
        vectors = dataset._pipeline_validation_vectors
        positions = dataset._pipeline_validation_positions
        # Positives may be absent with a partial pool. Extract these once as needed.
        needed = list(dict.fromkeys(dataset.qidxs + dataset.pidxs + [i for ns in dataset.nidxs for i in ns]))
        missing = [i for i in needed if i not in positions]
        if missing:
            extras = pipeline_extract_gpu(model, [dataset.images[i] for i in missing], dataset.imsize, dataset.transform)
            offset = vectors.shape[1]
            vectors = torch.cat([vectors, extras], dim=1)
            positions.update({i: offset+j for j, i in enumerate(missing)})
        # Original validation loader iterator consumes a CPU base seed.
        torch.empty((), dtype=torch.int64).random_()
        total_loss, count = 0.0, 0
        batch_size = loader.batch_size
        # Same non-shuffled tuples and drop_last as upstream; no image decode/forward again.
        stop = len(dataset) - len(dataset) % batch_size if loader.drop_last else len(dataset)
        for start in range(0, stop, batch_size):
            tuples = range(start, min(start+batch_size, stop))
            indices, labels = [], []
            for q in tuples:
                indices.extend(positions[i] for i in [dataset.qidxs[q], dataset.pidxs[q]] + dataset.nidxs[q])
                labels.extend([-1, 1] + [0]*dataset.nnum)
            target = torch.tensor(labels, dtype=torch.float32, device=vectors.device)
            total_loss += criterion(vectors[:, indices], target).item()
            count += len(tuples)
        print(f'Validation: reused mining descriptors for {count} tuples; no second image pass.')
        return total_loss / count
    finally:
        del dataset._pipeline_validation_vectors
        del dataset._pipeline_validation_positions

PIPELINE_IDENTITY = dict(settings=PIPELINE_SETTINGS, source_sha256='7d3f111599eca592d01d4ba3b05f10ebc3b1bdede4240f32570756906ebe3a09')
def preview_mining_pool(dataset):
    # Original method draws query permutation, then image permutation, from CPU RNG.
    # Replay both on a separate generator: no global RNG consumption or pool resampling.
    global_state = torch.get_rng_state().clone()
    preview_rng = torch.Generator(device='cpu')
    preview_rng.set_state(global_state)
    selected_pairs = torch.randperm(len(dataset.qpool), generator=preview_rng)[:dataset.qsize].tolist()
    sampled_pool = torch.randperm(len(dataset.images), generator=preview_rng)[:dataset.poolsize].tolist()
    query_indices = [int(dataset.qpool[i]) for i in selected_pairs]
    pool_clusters = {dataset.clusters[i] for i in sampled_pool}
    eligible_counts = [len(pool_clusters - {dataset.clusters[q]}) for q in query_indices]
    assert torch.equal(global_state, torch.get_rng_state()), 'Diagnostics altered the original RNG'
    diagnostic = dict(partition=dataset.mode, sampled_images=len(sampled_pool),
                      distinct_pool_clusters=len(pool_clusters),
                      minimum_eligible_negative_clusters=min(eligible_counts), queries=len(query_indices))
    return diagnostic, query_indices

class CheckedTuplesDataset(traindataset.TuplesDataset):
    def create_epoch_tuples(self, net):
        previous_stage = _session_state['stage']
        mining_started = time.monotonic()
        stage(f'Mining {self.mode}: queries={self.qsize}, pool={self.poolsize}')
        diagnostic, expected_queries = preview_mining_pool(self)
        self.last_mining_diagnostic = diagnostic
        print('Sampled mining pool:', json.dumps(diagnostic))
        if diagnostic['minimum_eligible_negative_clusters'] < self.nnum:
            raise RuntimeError(f'Insufficient sampled pool diversity: {diagnostic}. '
                               'Increase SMOKE_MINING_POOL_CAP for smoke, or full pool_size for training; '
                               'the original pool is never resampled automatically.')
        try:
            distance = pipeline_mine(self, net)  # Same tuple draws; unique descriptor extraction.
        except IndexError as error:
            raise RuntimeError('Original mining exhausted the sampled pool. '
                               'Increase the corresponding pool setting explicitly; no resampling performed.') from error
        self.last_mining_seconds = time.monotonic() - mining_started
        stage(previous_stage + f' | mining finished in {self.last_mining_seconds:.1f}s; processing tuples')
        assert [int(q) for q in self.qidxs] == expected_queries, 'Original query draw differed from diagnostic replay'
        assert math.isfinite(distance)
        assert len(self.qidxs) == len(self.pidxs) == len(self.nidxs) == self.qsize
        for q, p, negatives in zip(self.qidxs, self.pidxs, self.nidxs):
            assert self.clusters[q] == self.clusters[p]
            assert len(negatives) == 5 and len({int(i) for i in negatives}) == 5
            assert len({self.clusters[i] for i in negatives}) == 5
            assert self.clusters[q] not in {self.clusters[i] for i in negatives}
        print(f'Mining checked: {self.mode}, tuples={self.qsize}, q+p+5=7, pool={self.poolsize}, mean_ndist={distance:.6f}')
        return distance

def make_loaders(cfg):
    train_ds = CheckedTuplesDataset('retrieval-SfM-120k', 'train', imsize=cfg['image_size'],
                    nnum=5, qsize=cfg['query_size'], poolsize=cfg['pool_size'], transform=transform)
    val_ds = CheckedTuplesDataset('retrieval-SfM-120k', 'val', imsize=cfg['image_size'],
                    nnum=5, qsize=float('inf'), poolsize=float('inf'), transform=transform)
    kwargs = dict(batch_size=cfg['batch_size'], num_workers=effective_workers(cfg['workers']), pin_memory=True,
                  drop_last=True, collate_fn=datahelpers.collate_tuples)
    train_loader = torch.utils.data.DataLoader(train_ds, shuffle=True, **kwargs)
    val_loader = torch.utils.data.DataLoader(val_ds, shuffle=False, **kwargs)
    print('Actual train queries/pool:', train_ds.qsize, train_ds.poolsize)
    print('Validation pairs/pool/processed:', val_ds.qsize, val_ds.poolsize, len(val_loader)*cfg['batch_size'])
    return train_loader, val_loader

def make_optimizer(model, cfg):
    groups = [{'params': model.features.parameters()},
              {'params': model.pool.parameters(), 'lr': cfg['lr'] * 10, 'weight_decay': 0}]
    if cfg['optimizer'] == 'adam':
        return torch.optim.Adam(groups, lr=cfg['lr'], weight_decay=cfg['weight_decay'])
    return torch.optim.SGD(groups, lr=cfg['lr'], momentum=cfg['momentum'], weight_decay=cfg['weight_decay'])

def configure_original_loop(cfg):
    original_train.args = SimpleNamespace(update_every=cfg['update_every'], print_freq=10)
    original_train.train = pipeline_train
    original_train.validate = pipeline_validate
    imageretrievalnet.extract_vectors = pipeline_extract_vectors

# Legacy arithmetic sanity checks; no models or benchmark forwards.
assert abs(compute_ap(np.array([2]), 1) - 1/6) < 1e-12
score, _, _, _ = compute_map(np.array([[0], [1], [2], [3]]), [{'ok': [3], 'junk': [0, 1]}])
assert abs(score - 0.25) < 1e-12

imageretrievalnet.extract_vectors = pipeline_extract_vectors
ContrastiveLoss = PipelineContrastiveLoss


In [ ]:
SMOKE_RESULT = None

def run_smoke_test(cfg):
    # Function scope and finally cleanup keep failed GPU objects out of notebook globals.
    smoke_dir = WORKSPACE / 'smoke' / str(time.time_ns())
    smoke_dir.mkdir(parents=True, exist_ok=False)
    assert not smoke_dir.is_relative_to(WORKSPACE / 'runs'), 'Smoke must not share full-run storage'
    smoke_model = smoke_optimizer = smoke_loss = smoke_train = smoke_val = None
    before = imgs = labels = vectors = val_loss = restored = handle = None
    gradient_checks = []
    result = dict(status='FAILED', configuration=cfg, baseline_configuration=cfg,
                  source_commit=SOURCE_COMMIT, dataset_identity=DATASET_IDENTITY,
                  environment=ENVIRONMENT, log=str(smoke_dir / 'smoke.log'))
    start = time.monotonic()
    with logged(smoke_dir / 'smoke.log'):
        try:
            seed_all(cfg['seed'])
            reset_gpu_peaks()
            print('Smoke configuration:', json.dumps(cfg, indent=2))
            smoke_model = make_model(cfg['gem_p'])
            smoke_optimizer = make_optimizer(smoke_model, cfg)
            smoke_loss = ContrastiveLoss(margin=cfg['loss_margin']).cuda()
            smoke_train, smoke_val = make_loaders(cfg)
            configure_original_loop(cfg)
            before = smoke_model.features[0].weight.detach().clone()
            def check_gradients(optimizer, args, kwargs):
                grads = [p.grad for group in optimizer.param_groups for p in group['params'] if p.grad is not None]
                assert grads and all(torch.isfinite(g).all().item() for g in grads), 'Nonfinite gradients'
                gradient_checks.append(True)
            handle = smoke_optimizer.register_step_pre_hook(check_gradients)
            stage('Smoke: mining and one training batch at configured resolution')
            smoke_train_loss = original_train.train(smoke_train, smoke_model, smoke_loss, smoke_optimizer, 0)
            assert gradient_checks and math.isfinite(smoke_train_loss)
            assert not torch.equal(before, smoke_model.features[0].weight.detach()), 'No feature update'
            # One diagnostic validation tuple, original seven-image targets and loss.
            smoke_val.dataset.qsize = 1
            with torch.no_grad():
                smoke_val.dataset.create_epoch_tuples(smoke_model)
                imgs, labels = smoke_val.dataset[0]
                vectors = torch.cat([smoke_model(x.cuda()) for x in imgs], dim=1)
                val_loss = smoke_loss(vectors, labels.cuda())
            assert vectors.shape == (512, 7) and torch.isfinite(vectors).all()
            assert torch.allclose(vectors.norm(dim=0), torch.ones(7, device='cuda'), atol=1e-5)
            assert torch.isfinite(val_loss).item()
            # Separate smoke-only path, never latest.pt/best.pt or any full-run directory.
            smoke_checkpoint = smoke_dir / 'roundtrip.pt'
            temporary = smoke_dir / 'roundtrip.pt.tmp'
            torch.save({'state_dict': smoke_model.state_dict(), 'optimizer': smoke_optimizer.state_dict()}, temporary)
            temporary.replace(smoke_checkpoint)
            restored = torch.load(smoke_checkpoint, map_location='cpu', weights_only=True)
            assert all(torch.equal(v.cpu(), restored['state_dict'][k]) for k, v in smoke_model.state_dict().items())
            smoke_model.load_state_dict(restored['state_dict'], strict=True)
            smoke_optimizer.load_state_dict(restored['optimizer'])
            assert [g['lr'] for g in smoke_optimizer.param_groups] == [g['lr'] for g in restored['optimizer']['param_groups']]
            result.update(status='PASS', train_loss=smoke_train_loss, val_loss=val_loss.item(),
                          gradient_checks=len(gradient_checks), checkpoint=str(smoke_checkpoint),
                          train_mining=smoke_train.dataset.last_mining_diagnostic,
                          validation_mining=smoke_val.dataset.last_mining_diagnostic)
            print('SMOKE PASS: original mining, finite loss/gradients, parameter update and checkpoint round-trip checked.')
        except Exception as error:
            result.update(error_type=type(error).__name__, error=str(error))
            print('SMOKE FAILED:', result['error_type'], result['error'])
            traceback.print_exception(type(error), error, error.__traceback__)
            # Release upstream frames holding activation tensors; do not re-raise their traceback.
            traceback.clear_frames(error.__traceback__)
            error.__traceback__ = None
            print('No full training was started. Review this log; adjust the explicit pool cap for diversity failures. '
                  'For OOM, reduce resource settings explicitly or restart the kernel before retrying.')
        finally:
            if handle is not None:
                handle.remove()
            result['seconds'] = time.monotonic() - start
            result['gpu_memory_by_device'] = gpu_memory_stats()
            result['peak_gpu_bytes'] = torch.cuda.max_memory_allocated()
            result['cuda_allocated_before_cleanup'] = torch.cuda.memory_allocated()
            smoke_model = smoke_optimizer = smoke_loss = smoke_train = smoke_val = None
            before = imgs = labels = vectors = val_loss = restored = handle = None
            gc.collect()
            torch.cuda.empty_cache()
            result['cuda_allocated_after_cleanup'] = torch.cuda.memory_allocated()
            print('GPU allocated bytes after smoke cleanup:', result['cuda_allocated_after_cleanup'])
            (smoke_dir / 'result.json').write_text(json.dumps(result, indent=2, allow_nan=False))
    return result

AUTO_SMOKE = RUN_FULL_TRAINING  # New engine gets a smoke even on legacy resume.
if RUN_SMOKE or AUTO_SMOKE:
    if AUTO_SMOKE:
        smoke_cfg = dict(resolved_config('full'))
        assert isinstance(SMOKE_MINING_POOL_CAP, int) and not isinstance(SMOKE_MINING_POOL_CAP, bool)
        assert SMOKE_MINING_POOL_CAP >= 6
        smoke_cfg.update(query_size=smoke_cfg['batch_size'],
                         pool_size=min(SMOKE_MINING_POOL_CAP, len(SFM_DB['train']['cids'])),
                         epochs=1, update_every=1)
        print('Pipeline engine: automatic separate smoke at the full image size (including resume).')
    else:
        smoke_cfg = resolved_config('smoke')
    smoke_outcome = run_smoke_test(smoke_cfg)
    if smoke_outcome['status'] != 'PASS':
        # This new exception holds only a plain result dictionary, not model/activation frames.
        raise RuntimeError('Smoke failed; GPU cleanup completed. Details: ' + smoke_outcome['log']) from None
    SMOKE_RESULT = smoke_outcome
    print(json.dumps(SMOKE_RESULT, indent=2))
else:
    print('Smoke disabled. Configuration is ready; enable RUN_SMOKE explicitly after preflight.')


## 11–12. Full training controller, validation, logging, checkpoints, resume
RUN_FULL_TRAINING is True in this edited notebook. It runs the **fixed subset-100**, not the full SfM metadata. Fresh training must follow a successful smoke in this kernel; epoch-boundary resume uses an explicitly supplied trusted notebook checkpoint. The planned epoch count is a total target including completed epochs.

The controller preserves scheduler.step() before training and source loss/gradient accumulation. Checkpoints add scheduler state, RNG state, configuration, dataset identity, source commit/hashes and environment. Resume loads scheduler state rather than reconstructing/decaying it again. Only completed epochs are resumable; an interruption mid-epoch replays that epoch from the previous checkpoint. CUDA/library changes can prevent bitwise reproducibility.

Storage is bounded to latest.pt and best.pt, replaced atomically within this run folder. No unrelated files are deleted. A checkpoint is saved immediately after validation, before expensive benchmark evaluation, so evaluation failure does not discard the epoch. Best is chosen only by validation loss; benchmarks are never used to select it. Scheduler/seed changes and dropped residual tuples are listed in the run record. Finite gradients are checked before each update; no clipping or optimizer changes are introduced.


In [ ]:
def atomic_json(path, value):
    path = Path(path); temporary = path.with_suffix(path.suffix + '.tmp')
    temporary.write_text(json.dumps(value, indent=2, allow_nan=False))
    temporary.replace(path)

def atomic_checkpoint(path, state):
    path = Path(path); temporary = path.with_suffix(path.suffix + '.tmp')
    torch.save(state, temporary)
    temporary.replace(path)

def rng_state():
    return dict(python=random.getstate(), numpy=np.random.get_state(), torch=torch.get_rng_state(),
                cuda=torch.cuda.get_rng_state_all())

def restore_rng(state):
    random.setstate(state['python']); np.random.set_state(state['numpy'])
    torch.set_rng_state(state['torch'].cpu())
    torch.cuda.set_rng_state_all([s.cpu() for s in state['cuda']])

def config_identity(cfg):
    # Total epoch target may be extended on resume; all other choices must match.
    return {k: v for k, v in cfg.items() if k != 'epochs'}

def assert_checkpoint_identity(state, cfg=None):
    if 'pipeline_engine' in state:
        assert state['pipeline_engine'] == PIPELINE_IDENTITY, 'Pipeline changed since checkpoint'
    assert state.get('format') == 'b0_sfm100_notebook_v1', 'Use a checkpoint produced by this notebook'
    assert state['source_commit'] == SOURCE_COMMIT
    assert state['source_hashes'] == SOURCE_HASHES
    assert state['dataset_identity'] == DATASET_IDENTITY, 'Dataset/weights identity changed'
    if cfg is not None:
        assert config_identity(state['config']) == config_identity(cfg), 'Run configuration changed on resume'

def checkpoint_state(model, optimizer, scheduler, epoch, best_loss, cfg, history):
    return dict(format='b0_sfm100_notebook_v1', epoch=epoch, meta=model.meta,
                state_dict=model.state_dict(), optimizer=optimizer.state_dict(), scheduler=scheduler.state_dict(),
                min_loss=best_loss, rng=rng_state(), config=cfg, dataset_identity=DATASET_IDENTITY,
                source_commit=SOURCE_COMMIT, source_hashes=SOURCE_HASHES, environment=ENVIRONMENT, history=history, pipeline_engine=PIPELINE_IDENTITY)

# This is the original single-scale extraction/ranking/AP protocol, with returned metrics.
def evaluate_legacy(model, image_size, destination, label):
    result = dict(label=label, image_size=image_size, scale=[1], whitening=False, query_expansion=False,
                  protocol='legacy_ok_junk', source_commit=SOURCE_COMMIT, datasets={})
    for name in ['oxford5k', 'paris6k']:
        stage(f'Evaluation {label}: {name} gallery extraction (no training updates)')
        bench = configdataset(name, str(DATA_ROOT / 'test'))
        images = [bench['im_fname'](bench, i) for i in range(bench['n'])]
        queries = [bench['qim_fname'](bench, i) for i in range(bench['nq'])]
        boxes = [tuple(g['bbx']) for g in bench['gnd']]
        gallery = imageretrievalnet.extract_vectors(model, images, image_size, transform, ms=[1])
        stage(f'Evaluation {label}: {name} query extraction')
        query = imageretrievalnet.extract_vectors(model, queries, image_size, transform, bbxs=boxes, ms=[1])
        assert gallery.shape == (512, bench['n']) and query.shape == (512, bench['nq'])
        assert torch.isfinite(gallery).all() and torch.isfinite(query).all()
        assert torch.allclose(gallery.norm(dim=0), torch.ones(bench['n']), atol=1e-4)
        assert torch.allclose(query.norm(dim=0), torch.ones(bench['nq']), atol=1e-4)
        ranks = np.argsort(-(gallery.numpy().T @ query.numpy()), axis=0)
        map_score, aps, _, _ = compute_map(ranks, bench['gnd'])
        assert np.isfinite(map_score) and np.isfinite(aps).all()
        result['datasets'][name] = dict(map=float(map_score), map_percent=float(map_score*100), aps=aps.tolist())
        print(name, 'legacy mAP:', f'{map_score*100:.4f}%')
        del gallery, query, ranks
    atomic_json(destination, result)
    return result

TRAINING_RESULT = None
if RUN_FULL_TRAINING:
    cfg = resolved_config('full')
    assert re.fullmatch(r'[A-Za-z0-9_-]+', RUN_ID), 'RUN_ID must be a simple folder name'
    run_dir = WORKSPACE / 'runs' / RUN_ID
    if RESUME_CHECKPOINT is None:
        assert SMOKE_RESULT is not None and SMOKE_RESULT['status'] == 'PASS', 'Run a successful smoke before fresh full training'
        assert SMOKE_RESULT['dataset_identity'] == DATASET_IDENTITY
        assert SMOKE_RESULT['source_commit'] == SOURCE_COMMIT
        assert all(SMOKE_RESULT['baseline_configuration'][key] == cfg[key] for key in
                   ['image_size', 'batch_size', 'optimizer', 'lr', 'weight_decay', 'momentum', 'loss_margin', 'gem_p', 'seed', 'workers']), 'Core settings changed since smoke; repeat smoke'
        if run_dir.exists() and any(run_dir.iterdir()):
            raise RuntimeError('Run directory is not empty. Choose a new RUN_ID or resume explicitly.')
    else:
        if not Path(RESUME_CHECKPOINT).is_file():
            raise RuntimeError('Resume checkpoint is missing: ' + str(RESUME_CHECKPOINT))
        if run_dir.exists() and any(run_dir.iterdir()):
            local_latest = run_dir / 'latest.pt'
            if not local_latest.is_file() or Path(RESUME_CHECKPOINT).resolve() != local_latest.resolve():
                raise RuntimeError('For an imported checkpoint use a new RUN_ID; to continue this folder select its latest.pt.')
    run_dir.mkdir(parents=True, exist_ok=True)
    seed_all(cfg['seed'])
    model = make_model(cfg['gem_p'])
    optimizer = make_optimizer(model, cfg)
    scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer, gamma=math.exp(-0.01))
    criterion = ContrastiveLoss(margin=cfg['loss_margin']).cuda()
    train_loader, val_loader = make_loaders(cfg)
    configure_original_loop(cfg)
    start_epoch, best_loss, history = 0, float('inf'), []
    if RESUME_CHECKPOINT is not None:
        # Explicit pickle load is needed for our RNG state; use only your own trusted run artifacts.
        resumed = torch.load(RESUME_CHECKPOINT, map_location='cpu', weights_only=False)
        assert_checkpoint_identity(resumed, cfg)
        model.load_state_dict(resumed['state_dict'], strict=True)
        optimizer.load_state_dict(resumed['optimizer'])
        scheduler.load_state_dict(resumed['scheduler'])
        start_epoch, best_loss, history = resumed['epoch'], resumed['min_loss'], resumed['history']
        restore_rng(resumed['rng'])
        # Preserve the best state when continuing into a new run folder from an uploaded checkpoint.
        if not (run_dir / 'best.pt').exists():
            if resumed['epoch'] == min(history, key=lambda row: row['val_loss'])['epoch']:
                atomic_checkpoint(run_dir / 'best.pt', resumed)
            else:
                sibling_best = Path(RESUME_CHECKPOINT).parent / 'best.pt'
                if not sibling_best.is_file():
                    raise RuntimeError('Attach the matching best.pt alongside latest.pt to preserve model selection.')
                best_state = torch.load(sibling_best, map_location='cpu', weights_only=False)
                assert_checkpoint_identity(best_state, cfg)
                assert best_state['min_loss'] == best_loss
                assert best_state['epoch'] == min(history, key=lambda row: row['val_loss'])['epoch']
                atomic_checkpoint(run_dir / 'best.pt', best_state)
        if (run_dir / 'best.pt').exists():
            best_state = torch.load(run_dir / 'best.pt', map_location='cpu', weights_only=False)
            assert_checkpoint_identity(best_state, cfg)
            assert best_state['epoch'] == min(history, key=lambda row: row['val_loss'])['epoch']
            assert best_state['min_loss'] == best_loss
            del best_state
        del resumed
    assert 0 <= start_epoch < cfg['epochs'], 'Checkpoint already reaches epoch target; extend epochs or run final evaluation.'
    record = dict(config=cfg, decision_note=FULL_DECISION_NOTE, source_commit=SOURCE_COMMIT, source_hashes=SOURCE_HASHES,
                  dataset_identity=DATASET_IDENTITY, paths=PATHS, environment=ENVIRONMENT,
                  deviations_from_defaults={k: [SOURCE_DEFAULTS[k], v] for k,v in cfg.items() if v != SOURCE_DEFAULTS[k]},
                  actual_train_pool=train_loader.dataset.poolsize, actual_train_queries=train_loader.dataset.qsize,
                  processed_train_tuples=len(train_loader)*cfg['batch_size'], processed_val_tuples=len(val_loader)*cfg['batch_size'],
                  subset='fixed k100 seed42; active metadata only', resume=str(RESUME_CHECKPOINT),
                  scheduler_gamma=math.exp(-0.01), scheduler_timing='before train',
                  checkpoint_policy='atomic latest+best; epoch-boundary resume; checkpoint before benchmark eval',
                  mining_workers=effective_workers(8), precision='FP32', smoke=SMOKE_RESULT,
                  pipeline_engine=PIPELINE_IDENTITY, prepared_dataset=str(PREPARED_DIR), operational_revision='pipeline_v2', console_progress_seconds=CONSOLE_PROGRESS_SECONDS,
                  effective_train_workers=effective_workers(cfg['workers']),
                  evaluation_workers=effective_workers(8), requested_train_workers=cfg['workers'])
    atomic_json(run_dir / 'run_record.json', record)
    def finite_gradients(optimizer, args, kwargs):
        grads = [p.grad for group in optimizer.param_groups for p in group['params'] if p.grad is not None]
        # One CPU synchronization per device instead of one per parameter tensor.
        flags_by_device = {}
        for grad in grads:
            flags_by_device.setdefault(grad.device, []).append(torch.isfinite(grad).all())
        if not grads or not all(torch.stack(flags).all().item() for flags in flags_by_device.values()):
            raise FloatingPointError('Nonfinite/absent gradients; optimizer update aborted')
    grad_handle = optimizer.register_step_pre_hook(finite_gradients)
    try:
        with logged(run_dir / 'training.log'):
            print('Starting/resuming epoch:', start_epoch, 'LR:', [g['lr'] for g in optimizer.param_groups])
            if start_epoch == 0:
                print('Initial Oxford/Paris evaluation comes BEFORE epoch 1; this can take a while.', flush=True)
                evaluate_legacy(model, cfg['eval_image_size'], run_dir / 'evaluation_initial.json', 'initial_caffe')
            for epoch in range(start_epoch, cfg['epochs']):
                seed_all(cfg['seed'] + epoch)
                start = time.monotonic()
                reset_gpu_peaks()
                scheduler.step()  # Preserve the historical timing; modern torch may warn on first step.
                learning_rates = [g['lr'] for g in optimizer.param_groups]
                print('Epoch', epoch+1, 'seed', cfg['seed']+epoch, 'LR before update', learning_rates)
                stage(f'Epoch {epoch+1}: training including hard-negative mining')
                train_started = time.monotonic()
                train_loss = original_train.train(train_loader, model, criterion, optimizer, epoch)
                train_seconds = time.monotonic() - train_started
                validation_started = time.monotonic()
                stage(f'Epoch {epoch+1}: validation including mining')
                with torch.no_grad():
                    val_loss = original_train.validate(val_loader, model, criterion, epoch)
                validation_seconds = time.monotonic() - validation_started
                assert math.isfinite(train_loss) and math.isfinite(val_loss)
                improved = val_loss < best_loss
                best_loss = min(best_loss, val_loss)
                history.append(dict(epoch=epoch+1, train_loss=train_loss, val_loss=val_loss,
                                    lr=learning_rates, gem_p=model.pool.p.item(), seconds_train_val=time.monotonic()-start,
                                    peak_gpu_bytes=torch.cuda.max_memory_allocated(),
                                    gpu_memory_by_device=gpu_memory_stats()))
                stage(f'Epoch {epoch+1}: saving checkpoint')
                checkpoint_started = time.monotonic()
                state = checkpoint_state(model, optimizer, scheduler, epoch+1, best_loss, cfg, history)
                atomic_checkpoint(run_dir / 'latest.pt', state)
                if improved:
                    atomic_checkpoint(run_dir / 'best.pt', state)
                atomic_json(run_dir / 'history.json', history)
                checkpoint_seconds = time.monotonic() - checkpoint_started
                del state  # Release checkpoint references before benchmark extraction.
                print('Checkpoint saved, best validation loss:', best_loss)
                benchmark_started = time.monotonic()
                if (epoch+1) % cfg['test_freq'] == 0:
                    evaluate_legacy(model, cfg['eval_image_size'], run_dir / f'evaluation_epoch_{epoch+1:03d}.json', f'epoch_{epoch+1}')
                benchmark_seconds = time.monotonic() - benchmark_started
                timings = dict(epoch=epoch+1, train_including_mining_seconds=train_seconds,
                    train_mining_seconds=train_loader.dataset.last_mining_seconds,
                    train_without_mining_seconds=max(0, train_seconds-train_loader.dataset.last_mining_seconds),
                    validation_including_mining_seconds=validation_seconds,
                    validation_mining_seconds=val_loader.dataset.last_mining_seconds,
                    checkpoint_seconds=checkpoint_seconds, benchmark_seconds=benchmark_seconds,
                    total_seconds=time.monotonic()-start)
                atomic_json(run_dir / f'timing_epoch_{epoch+1:03d}.json', timings)
                print('Epoch timing:', json.dumps(timings))
            TRAINING_RESULT = dict(completed_epochs=len(history), best_validation_loss=best_loss,
                                   latest=str(run_dir / 'latest.pt'), best=str(run_dir / 'best.pt'))
            atomic_json(run_dir / 'training_result.json', TRAINING_RESULT)
    finally:
        grad_handle.remove()
        del model, optimizer, scheduler, criterion, train_loader, val_loader
        gc.collect(); torch.cuda.empty_cache()
else:
    print('Full training disabled. No optimizer updates or benchmark evaluation are launched by this cell.')


## 13. Final legacy mAP evaluation — explicit opt-in
Enable RUN_FINAL_EVALUATION to evaluate a trusted notebook-generated checkpoint. By default it loads the run's **best validation-loss** checkpoint, not the last model or smoke model. A selected path is permitted under /kaggle/input for a resumed/offline session. Both benchmarks use original query bounding boxes and gallery order, dot-product ranking and upstream trapezoidal AP with junk removal. The checkpoint SHA-256 is saved alongside results. No whitening or QE code is called.


In [ ]:
FINAL_RESULTS = None
if RUN_FINAL_EVALUATION:
    assert re.fullmatch(r'[A-Za-z0-9_-]+', RUN_ID)
    checkpoint = Path(EVALUATION_CHECKPOINT) if EVALUATION_CHECKPOINT else WORKSPACE / 'runs' / RUN_ID / 'best.pt'
    if not checkpoint.is_file():
        raise RuntimeError('No trained checkpoint available for final evaluation: ' + str(checkpoint))
    state = torch.load(checkpoint, map_location='cpu', weights_only=False)  # Trusted notebook artifact only.
    assert_checkpoint_identity(state)
    evaluation_cfg = state['config']
    eval_model = make_model(evaluation_cfg['gem_p'])
    eval_model.load_state_dict(state['state_dict'], strict=True)
    out = WORKSPACE / 'evaluations' / (RUN_ID + '_' + str(time.time_ns()))
    out.mkdir(parents=True)
    try:
        with logged(out / 'evaluation.log'):
            FINAL_RESULTS = evaluate_legacy(eval_model, evaluation_cfg['eval_image_size'], out / 'metrics.json',
                                            'checkpoint_epoch_' + str(state['epoch']))
            FINAL_RESULTS.update(checkpoint=str(checkpoint), checkpoint_sha256=sha256_file(checkpoint),
                                 checkpoint_epoch=state['epoch'], dataset_identity=DATASET_IDENTITY, environment=ENVIRONMENT)
            atomic_json(out / 'metrics.json', FINAL_RESULTS)
    finally:
        del state, eval_model
        gc.collect()
        torch.cuda.empty_cache()

else:
    print('Final evaluation disabled; no mAP result is claimed.')


In [ ]:
print('Workspace:', WORKSPACE)
print('Source commit:', SOURCE_COMMIT)
print('Dataset metadata SHA-256:', DATASET_IDENTITY['metadata_sha256'])
print('Smoke result:', SMOKE_RESULT)
print('Training result:', TRAINING_RESULT)
print('Final result:', FINAL_RESULTS)
for folder in ['runs', 'evaluations']:
    root = WORKSPACE / folder
    if root.exists():
        for path in sorted(root.rglob('*')):
            if path.is_file():
                print(path.relative_to(WORKSPACE), path.stat().st_size, 'bytes')
